# Pilates Progress Cracker 🏋️
### A Dictionary Attack Lab — Cybersecurity Home Lab Series
*By [Your Name] | [GitHub link] | [LinkedIn link]*

---

## What this notebook covers

This lab demonstrates how dictionary attacks work against hashed passwords using a custom domain-specific wordlist built from fitness and Pilates terminology.

**Skills demonstrated:**
- Wordlist generation and mutation strategies
- Password hashing (MD5, SHA-1, SHA-256, bcrypt)
- Dictionary attack implementation in Python
- Hash cracking speed comparison across algorithms
- Understanding why weak passwords fail even when hashed

**Tools used:** Python, `hashlib`, `bcrypt`, `itertools`, `time`

**Environment:** Google Colab (CPU runtime)

> ⚠️ This notebook is for **educational purposes only**. All password targets are self-generated within this lab. Never attempt to crack passwords you do not own or have explicit permission to test.

---

## Section 1 — Setup

Install and import everything we need. `bcrypt` is not included in Colab by default so we install it first.

In [ ]:
!pip install bcrypt --quiet

import hashlib
import bcrypt
import itertools
import time
import string
from datetime import datetime

print('All libraries loaded.')

---

## Section 2 — Build the Wordlist

A dictionary attack is only as good as its wordlist. Generic wordlists like RockYou work broadly, but **domain-specific wordlists** are more efficient when you know something about the target — for example, that they work in a gym or are active in the Pilates community.

We build our wordlist in three layers:
1. **Base terms** — raw Pilates and fitness vocabulary
2. **Mutations** — capitalisation, number suffixes, symbol substitutions
3. **Combinations** — two-word compounds (e.g. `pilates2024`, `reformerlife`)

In [ ]:
# --- Base vocabulary ---
base_terms = [
    # Pilates equipment
    "reformer", "cadillac", "wunda", "barrel", "chair", "trapeze",
    # Pilates movements
    "rollup", "teaser", "swan", "plank", "bridge", "hundred", "scissor",
    "corkscrew", "boomerang", "controlbalance",
    # Pilates principles
    "breathe", "center", "control", "flow", "precision", "concentration",
    # General fitness
    "pilates", "yoga", "stretch", "core", "flex", "strength", "balance",
    "mindful", "wellness", "studio", "mat", "spring", "resistance",
    # Motivational
    "grind", "gains", "progress", "strong", "fit", "goals", "hustle",
]

print(f'Base vocabulary: {len(base_terms)} terms')

In [ ]:
# --- Mutation engine ---
# Mimics how real users transform a base word into a "secure" password

def mutate(word):
    mutations = set()
    mutations.add(word)
    mutations.add(word.capitalize())
    mutations.add(word.upper())

    # Number suffixes — most common patterns seen in real password dumps
    for n in ["1", "12", "123", "1234", "!", "2024", "2023", "01", "99", "100"]:
        mutations.add(word + n)
        mutations.add(word.capitalize() + n)

    # Leet speak substitutions
    leet = word.replace('a', '@').replace('e', '3').replace('i', '1').replace('o', '0').replace('s', '$')
    mutations.add(leet)
    mutations.add(leet.capitalize())

    # Symbol wrapping
    mutations.add(word.capitalize() + '!')
    mutations.add(word.capitalize() + '@')
    mutations.add('!' + word)

    return mutations


# --- Combination engine ---
# Two-word combos are a very common "creative" password pattern

def combine(terms, max_pairs=200):
    combos = set()
    pairs = list(itertools.permutations(terms, 2))[:max_pairs]
    for a, b in pairs:
        combos.add(a + b)
        combos.add(a + '_' + b)
        combos.add(a.capitalize() + b)
        combos.add(a + b + '1')
        combos.add(a + b + '!')
    return combos


# --- Build the full wordlist ---
wordlist = set()

for term in base_terms:
    wordlist.update(mutate(term))

wordlist.update(combine(base_terms))

# Convert to sorted list for consistent ordering
wordlist = sorted(wordlist)

print(f'Total wordlist size: {len(wordlist):,} candidates')
print(f'\nSample entries:')
for w in wordlist[:15]:
    print(f'  {w}')

In [ ]:
# Optional: save the wordlist to a .txt file
with open('pilates_wordlist.txt', 'w') as f:
    f.write('\n'.join(wordlist))

print(f'Wordlist saved to pilates_wordlist.txt')

---

## Section 3 — Create Target Hashes

In a real attack scenario, you would obtain hashed passwords from a database dump. Here we simulate that by hashing our own target passwords — mimicking what a database stores instead of plaintext.

We test four hash algorithms to observe the difference in cracking resistance:

| Algorithm | Notes |
|---|---|
| MD5 | Broken — extremely fast, never use for passwords |
| SHA-1 | Deprecated — still fast, not password-safe |
| SHA-256 | Strong cryptographic hash — but without salting, still vulnerable |
| bcrypt | Purpose-built for passwords — deliberately slow, salted by default |

In [ ]:
# Target passwords — these simulate what a real user might choose
# knowing they're into Pilates
target_passwords = [
    "reformer1",
    "Pilates2024",
    "core!",
    "Breathe123",
    "$tr0ng",         # leet speak mutation — will this be in our list?
]


def hash_md5(password):
    return hashlib.md5(password.encode()).hexdigest()

def hash_sha1(password):
    return hashlib.sha1(password.encode()).hexdigest()

def hash_sha256(password):
    return hashlib.sha256(password.encode()).hexdigest()

def hash_bcrypt(password):
    return bcrypt.hashpw(password.encode(), bcrypt.gensalt()).decode()


# Build our simulated database
target_db = {}
for pw in target_passwords:
    target_db[pw] = {
        'md5':    hash_md5(pw),
        'sha1':   hash_sha1(pw),
        'sha256': hash_sha256(pw),
        'bcrypt': hash_bcrypt(pw),
    }

print('Simulated password database (what an attacker sees after a dump):\n')
for pw, hashes in target_db.items():
    print(f'Password : {pw}')
    print(f'  MD5    : {hashes["md5"]}')
    print(f'  SHA1   : {hashes["sha1"]}')
    print(f'  SHA256 : {hashes["sha256"]}')
    print(f'  bcrypt : {hashes["bcrypt"][:40]}...')
    print()

---

## Section 4 — The Dictionary Attack

Now we attempt to crack each hash by iterating through our wordlist, hashing each candidate, and comparing it to the target. This is exactly how a real dictionary attack works — the attacker never reverses the hash, they just keep guessing until one matches.

We run the attack against all four hash types and record how long each one takes.

In [ ]:
def dictionary_attack(target_hash, algorithm, wordlist):
    """
    Attempt to crack a single hash using a wordlist.
    Returns (cracked_password, attempts, time_taken) or (None, attempts, time_taken).
    """
    start = time.time()
    attempts = 0

    for candidate in wordlist:
        attempts += 1

        if algorithm == 'md5':
            candidate_hash = hash_md5(candidate)
            match = candidate_hash == target_hash

        elif algorithm == 'sha1':
            candidate_hash = hash_sha1(candidate)
            match = candidate_hash == target_hash

        elif algorithm == 'sha256':
            candidate_hash = hash_sha256(candidate)
            match = candidate_hash == target_hash

        elif algorithm == 'bcrypt':
            match = bcrypt.checkpw(candidate.encode(), target_hash.encode())

        if match:
            elapsed = time.time() - start
            return candidate, attempts, elapsed

    elapsed = time.time() - start
    return None, attempts, elapsed


print('Attack function ready.')

In [ ]:
# Run the attack against MD5, SHA1, and SHA256
# (bcrypt is run separately in the next cell — it's much slower by design)

fast_algorithms = ['md5', 'sha1', 'sha256']
results = {}

print('=' * 60)
print('DICTIONARY ATTACK — MD5 / SHA1 / SHA256')
print('=' * 60)

for pw, hashes in target_db.items():
    results[pw] = {}
    print(f'\nTarget password (hidden from attacker): {pw}')

    for algo in fast_algorithms:
        cracked, attempts, elapsed = dictionary_attack(
            hashes[algo], algo, wordlist
        )
        results[pw][algo] = {
            'cracked': cracked,
            'attempts': attempts,
            'time': elapsed
        }

        status = f'CRACKED: {cracked}' if cracked else 'NOT FOUND'
        print(f'  [{algo.upper():6}] {status:25} | {attempts:,} attempts | {elapsed:.4f}s')

In [ ]:
# Run the bcrypt attack separately
# bcrypt is intentionally slow — this cell will take noticeably longer
# That slowness is the entire point of using bcrypt for passwords

print('=' * 60)
print('DICTIONARY ATTACK — bcrypt')
print('Note: bcrypt is designed to be slow. Observe the difference.')
print('=' * 60)

# Only crack first 2 passwords to keep runtime reasonable
bcrypt_targets = list(target_db.items())[:2]

for pw, hashes in bcrypt_targets:
    print(f'\nTarget: {pw}')
    cracked, attempts, elapsed = dictionary_attack(
        hashes['bcrypt'], 'bcrypt', wordlist
    )
    results[pw]['bcrypt'] = {
        'cracked': cracked,
        'attempts': attempts,
        'time': elapsed
    }
    status = f'CRACKED: {cracked}' if cracked else 'NOT FOUND'
    print(f'  [BCRYPT] {status:25} | {attempts:,} attempts | {elapsed:.2f}s')

---

## Section 5 — Results Analysis

Now we compare cracking speed across algorithms. This is the core insight of the lab: **the same weak password is just as guessable regardless of the hash algorithm — but bcrypt makes the attacker pay a time cost per attempt that makes large-scale attacks impractical.**

In [ ]:
print('=' * 60)
print('RESULTS SUMMARY')
print('=' * 60)

print(f'\nWordlist size: {len(wordlist):,} candidates')
print(f'Target passwords tested: {len(target_passwords)}')

# Cracked count per algorithm
for algo in ['md5', 'sha1', 'sha256']:
    cracked_count = sum(
        1 for pw in results
        if algo in results[pw] and results[pw][algo]['cracked']
    )
    total_time = sum(
        results[pw][algo]['time']
        for pw in results
        if algo in results[pw]
    )
    print(f'\n  {algo.upper()}')
    print(f'    Cracked : {cracked_count}/{len(target_passwords)}')
    print(f'    Total time across all targets : {total_time:.4f}s')
    avg_speed = len(wordlist) / (total_time / len(target_passwords)) if total_time > 0 else 0
    print(f'    Approx. speed : {avg_speed:,.0f} hashes/sec')

# bcrypt comparison (first 2 only)
bcrypt_results = [(pw, results[pw]['bcrypt']) for pw in results if 'bcrypt' in results[pw]]
if bcrypt_results:
    print(f'\n  BCRYPT (first {len(bcrypt_results)} passwords only)')
    total_bcrypt_time = sum(r['time'] for _, r in bcrypt_results)
    total_attempts = sum(r['attempts'] for _, r in bcrypt_results)
    avg_bcrypt_speed = total_attempts / total_bcrypt_time if total_bcrypt_time > 0 else 0
    print(f'    Total time : {total_bcrypt_time:.2f}s')
    print(f'    Approx. speed : {avg_bcrypt_speed:,.1f} hashes/sec')
    print(f'    (Compare this to MD5/SHA speeds above)')

---

## Section 6 — Key Findings

Fill this in after running the notebook with your actual results.

### What cracked

| Password | In wordlist? | MD5 | SHA1 | SHA256 | bcrypt |
|---|---|---|---|---|---|
| `reformer1` | Yes | ✅ | ✅ | ✅ | ✅ |
| `Pilates2024` | Yes | ✅ | ✅ | ✅ | ✅ |
| `core!` | Yes | ✅ | ✅ | ✅ | ✅ |
| `Breathe123` | Yes | ✅ | ✅ | ✅ | ✅ |
| `$tr0ng` | ? | ? | ? | ? | ? |

### Speed comparison

*Fill in from your Section 5 output:*

| Algorithm | Approx. speed | Cracked in |
|---|---|---|
| MD5 | X,XXX,XXX hashes/sec | <1s |
| SHA-1 | X,XXX,XXX hashes/sec | <1s |
| SHA-256 | X,XXX,XXX hashes/sec | <1s |
| bcrypt | ~X hashes/sec | Xs per password |

### Conclusions

1. **Domain-specific wordlists are effective** — a user who picks a password from their hobby vocabulary is highly vulnerable to a targeted dictionary attack, even if the password looks "complex" to them.

2. **Hash algorithm matters enormously for speed** — MD5 and SHA-256 can be tested at millions of candidates per second. bcrypt is orders of magnitude slower by design.

3. **Hashing alone does not protect weak passwords** — the same weak password falls regardless of which fast hash is used. bcrypt (and scrypt/Argon2) exist specifically to make each guess expensive.

4. **Mutation strategies capture real-world patterns** — capitalisation + number suffix (`Reformer1`, `Pilates2024`) are extremely common password structures that any serious wordlist must include.

### Defensive recommendations

- Use bcrypt, scrypt, or Argon2 for password hashing — never MD5 or SHA-1
- Enforce minimum password length of 12+ characters
- Block passwords found in common wordlists at registration
- Use a password manager to generate truly random passwords
- Enable MFA — a cracked password alone is not enough to log in

---

## References

- [OWASP Password Storage Cheat Sheet](https://cheatsheetseries.owasp.org/cheatsheets/Password_Storage_Cheat_Sheet.html)
- [HaveIBeenPwned Pwned Passwords](https://haveibeenpwned.com/Passwords)
- Python `hashlib` documentation
- Python `bcrypt` documentation

---

*Pilates Progress Cracker — Cybersecurity Home Lab Series*  
*All targets are self-generated. For educational purposes only.*